***
# AirBnB Listings in Zurich: Data analysis
***
Before delving into the analysis of Airbnb listings in the city of Zurich, let's iinstall all necessary libraries:

In [ ]:
import pandas as pd

## 1. About the Project
### 1.1. AirBnB listings as a topic
The short-term rental market, AirBnB in particular, has grown rapidly in urban areas, influencing rents, local economies, and urban planning. In cities like Zürich, understanding the factors of Airbnb prices can provide insights into market dynamics, potential regulatory interventions, and correlations with traditional rental prices. 
### 1.2. The Datasets
Inside AirBnB, the provider of our main dataset, is a non-commercial third-party provider of AirBnB data that aims to increase transparency of Airbnb activity. They scrape the official Airbnb website regularly and structure it csv files which can be obtained via their website https://insideairbnb.com/.
In addition, we are using two other datasets, one for normalizing the airbnb listings over neighborhoods in Zurich and the other to compare these normalized listing quantities with rental prices in each neighborhood.
### 1.3. Our Goals
In our project we aim to analyse Airbnb listings in Zürich, identify key features influencing pricing, and compare them with municipal rental statistics.
## 2. Loading, Cleaning and Validating the datasets
### 2.1. Loading and displaying dataframe overviews


In [ ]:
# define an overview function
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "missing_pct"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    return summary

#loading and displaying
## Airbnb
listings_path = 'data/listings.csv'
airbnb_df= pd.read_csv(listings_path, encoding="utf-8")
display(overview(airbnb_df, "Airbnb"))

## housing stock
housing_path = 'data/bau522od5221_wohnungsbestand_zurich.csv'
housing_df = pd.read_csv(housing_path, encoding="utf-8")
display(overview(housing_df, "Housing"))

## Rental prices
rental_path = 'data/rental_prices.csv'
rental_df = pd.read_csv(rental_path, encoding="utf-8")
display(overview(rental_df,"Rental"))

### 2.2. Cleaning
Based on the overview, we have decided to clean the datasets the following way:
- Generally:
    - Standardize column names (lowercase, underscores etc.).
    - Turn every column with 2 unique values to boolean.
    - All object data types are converted to more primitive dtypes (string, float, int, category, sorted category).
- Airbnb:
    - Drop unnecessary columns (i.e. everything with a URL).
    - Remove columns with too much unusable noise, like *host_description* in *airbnb_df*. Instead, we added a column that states the length of the entry.
    - Reference all dates or temporal variables to the date the data was scraped.
    - Feature engineering amenities in teh Airbnb dataframe by extracting amenities from the list in the *amenities* column.
- Housing Stock:
    - Only keep entries from 2025
    - Restructure the dataframe to only have 32 rows(one for each neighborhood). Aggregate the number of objects into separate columns for different apartment sizes.
- Rental Prices:
    - Only keep entries for 2024 (remove 2022)
    - Only keep square meter entries
    - Only keep netto entries (to remove differences in accounting etc. between neighborhoods as much as possible)
    - Restructure the dataframe to only have 32 rows (one for each neighborhood). The columns now represent different rental prices.

A unique normalization function was defined for each of the 3 datasets in the file *normalize.py*. Consult this file for further detail on the normalization process.

In [ ]:
from src.normalize import normalize_airbnb, normalize_housing, normalize_rental

# normalize Airbnb df
airbnb_df_norm = normalize_airbnb(airbnb_df)

# normalize housing stock df
housing_df_norm = normalize_housing(housing_df, year=2025)

# normalize rental price df
rental_df_norm = normalize_rental(rental_df, year=2024, brutto=False, sqm=True, level = 5, cat_zimmer=False)

## 3. Aggregating to quartier-level
At this point, the housing stock dataframe and rental price dataframe contain hundreds of rows, where each Quartier has several entries. For our analysis on quartier-level, we prefer a different structure, where each quartier only has one row. For that reson, we create a new pivot table around the quartier-column.
### 3.1. Using Pivot Tables

In [ ]:
from src.aggregation import quartiere_standardized_housing, quartiere_standardized_rental

# housing stock
housing_df_quart = quartiere_standardized_housing(housing_df_norm)
# flatten columns of housing df to 1:
housing_df_quart.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in housing_df_quart.columns
]
housing_df_quart = housing_df_quart.drop(columns="index_")
display(housing_df_quart.head())

# rental prices
rental_df_quart = quartiere_standardized_rental(rental_df_norm)
display(rental_df_quart.head())

Now, the two datasets are of length 35 (i.e. both only contain one row for each of the 34 quartiere).
### 3.2. Add Airbnb count to Housing Stock dataframe
In order to compare airbnb listings (normalized by housing stock) and rental prices in each quartier, we need to know how many listings are in each quartier. Luckily, INside Airbnb provides the quartier that each listing lies in, so we simply need to count them and merge that count to *housing_df_quart*:

In [ ]:
# count airbnbs per quartier
counts_airbnb_quartier = (
    airbnb_df_norm
    .groupby("neighbourhood_cleansed")
    .size()
    .rename("n_airbnbs")
)

# merge
housing_df_quart = housing_df_quart.merge(
    counts_airbnb_quartier,
    left_on="quarlang_",
    right_index=True,
    how="left"
)
